# unbox-args-tensor-to-array — faded example 2: Unbox kwargs values

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `unbox-args-tensor-to-array`. Running the beacon reports progress on the `Backprop: Unbox Tensor args to array` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Unbox Tensor args to array` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`unbox-args-tensor-to-array`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "unbox-args-tensor-to-array"
DD_SUBTOPIC = "Backprop: Unbox Tensor args to array"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Unboxing a kwargs dict replaces each `MiniTensor` value with its `.array`, preserving keys and insertion order and returning a fresh dict. Non-Tensor values pass through; the `isinstance` gate prevents accidentally unwrapping raw arrays.

## Faded exercise 2

### Unbox the kwargs values

Implement `unbox_kwargs(kwargs)` returning a new dict where each `MiniTensor` value becomes its `.array` and other values pass through, keys and order preserved. Complete the value expression in the dict comprehension.

**Fill in:** the value expression: v.array if v is a MiniTensor else v

In [ ]:
class MiniTensor:
    def __init__(self, array):
        self.array = array

def unbox_kwargs(kwargs: dict) -> dict:
    return {k: (v.array if isinstance(v, MiniTensor) else v) for k, v in kwargs.items()}

print(unbox_kwargs({'src': MiniTensor([1]), 'dim': 0}))


def _test():
    src = MiniTensor([1, 2])
    kw = {'src': src, 'dim': 1, 'name': 'op'}
    out = unbox_kwargs(kw)
    assert isinstance(out, dict)
    # keys preserved, order preserved
    assert list(out.keys()) == ['src', 'dim', 'name']
    # tensor value unboxed to exact object; scalars pass through
    assert out['src'] is src.array
    assert out['dim'] == 1
    assert out['name'] == 'op'
    # original dict not mutated
    assert kw['src'] is src
    # empty -> empty
    assert unbox_kwargs({}) == {}


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
class MiniTensor:
    def __init__(self, array):
        self.array = array

def unbox_kwargs(kwargs: dict) -> dict:
    return {k: (v.array if isinstance(v, MiniTensor) else v) for k, v in kwargs.items()}

print(unbox_kwargs({'src': MiniTensor([1]), 'dim': 0}))
```
</details>